In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient("database", 27017)

# 2. Crear (o usar) la base de datos y la colección
db = client['ArriendosPrueba_IVRegión'] 
coleccion = db['G2_PuebaChilePropiedades'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [4]:
# ==============================================================================
# PROYECTO BIG DATA - GRUPO 2 (REAL ESTATE)
# Módulo de Extracción Profunda: ChilePropiedades (Arriendos IV Región)
# ==============================================================================

import os
import time
import re
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

# --- 🎯 ASIGNACIÓN DE PÁGINAS ---
PAGINA_INICIO = 1
PAGINA_FIN = 2

os.environ["DISPLAY"] = ":99"
os.system("pkill -9 chrome && pkill -9 chromedriver")

print(f"🧹 Entorno listo. ChilePropiedades: Páginas {PAGINA_INICIO} a {PAGINA_FIN}")

options = Options()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

catalogo_urls = []
datos_finales = []

try:
    driver = webdriver.Chrome(options=options)

    # ==============================================================================
    # FASE 1: RECOLECCIÓN
    # ==============================================================================

    url_base = "https://chilepropiedades.cl/propiedades/arriendo-mensual/departamento/coquimbo/"

    for pagina_actual in range(PAGINA_INICIO, PAGINA_FIN + 1):

        print(f"\n--- 📄 Página {pagina_actual} ---")

        url_pagina = f"{url_base}?pagina={pagina_actual}"

        driver.get(url_pagina)

        time.sleep(5)

        # Scroll para cargar contenido dinámico
        for _ in range(3):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

        print("Página cargada:", driver.title)

        links = driver.find_elements(By.TAG_NAME, "a")

        print("Links encontrados:", len(links))

        for link in links:
            try:
                url = link.get_attribute("href")

                if not url:
                    continue

                if "/propiedades/" not in url:
                    continue

                if "arriendo" not in url:
                    continue

                catalogo_urls.append({
                    "url": url,
                    "ubicacion": "Coquimbo"
                })

            except:
                continue

    # eliminar duplicados
    catalogo_urls = list({v['url']: v for v in catalogo_urls}.values())

    print(f"\nCatálogo listo: {len(catalogo_urls)} propiedades")


    # ==============================================================================
    # FASE 2: INSPECCIÓN
    # ==============================================================================

    for i, propiedad in enumerate(catalogo_urls):

        print(f"Procesando {i+1}/{len(catalogo_urls)}")

        registro = {
            "responsable": "Constanza Torres",
            "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M:%S"),
            "titulo": "",
            "ubicacion": propiedad["ubicacion"],
            "m2": 0,
            "precio": 0.0,
            "dormitorios": 0,
            "baños": 0,
            "estacionamiento": 0,
            "piscina": 0,
            "quincho": 0,
            "terraza": 0,
            "gimnasio": 0,
            "lavanderia": 0,
            "url": propiedad["url"]
        }

        try:

            driver.get(propiedad["url"])

            time.sleep(4)

            texto_total = driver.find_element(By.TAG_NAME, "body").text.lower()

            # titulo
            try:
                registro["titulo"] = driver.find_element(By.TAG_NAME, "h1").text
            except:
                pass

            # precio
            match_precio = re.search(r'\$\s?([\d\.]+)', texto_total)
            if match_precio:
                precio = match_precio.group(1).replace(".", "")
                registro["precio"] = float(precio)

            # m2
            match_m2 = re.search(r'(\d+)\s*(m2|m²)', texto_total)
            if match_m2:
                registro["m2"] = int(match_m2.group(1))

            # dormitorios
            match_dorm = re.search(r'(\d+)\s*(dormitorio|habitacion)', texto_total)
            if match_dorm:
                registro["dormitorios"] = int(match_dorm.group(1))

            # baños
            match_bano = re.search(r'(\d+)\s*baño', texto_total)
            if match_bano:
                registro["baños"] = int(match_bano.group(1))

            # estacionamiento
            match_est = re.search(r'(\d+)\s*estacionamiento', texto_total)
            if match_est:
                registro["estacionamiento"] = int(match_est.group(1))
            elif "estacionamiento" in texto_total:
                registro["estacionamiento"] = 1

            # amenidades
            if "piscina" in texto_total:
                registro["piscina"] = 1

            if "quincho" in texto_total:
                registro["quincho"] = 1

            if "terraza" in texto_total:
                registro["terraza"] = 1

            if "gimnasio" in texto_total:
                registro["gimnasio"] = 1

            if "lavander" in texto_total:
                registro["lavanderia"] = 1


            # FILTRO ARRIENDO DIARIO (más preciso)
            if "por noche" in texto_total or "por día" in texto_total:
                print("Descartado arriendo diario")
                continue


            # FILTRO PRECIO MUY BAJO
            if registro["precio"] != 0 and registro["precio"] < 200000:
                print("Descartado precio muy bajo")
                continue


            datos_finales.append(registro)

        except Exception as e:
            print("Error en propiedad:", e)

        time.sleep(2)

except Exception as e:
    print("Error general:", e)

finally:
    driver.quit()


# ==============================================================================
# GUARDADO MONGO
# ==============================================================================

print("Guardando Mongo")

client = MongoClient("database",27017)

db = client["ArriendosPrueba_IVRegión"]

coleccion = db["G2_PruebaChilePropiedades"]

for d in datos_finales:
    coleccion.update_one(
        {"url": d["url"]},
        {"$set": d},
        upsert=True
    )

print("Extracción terminada:", len(datos_finales))

🧹 Entorno listo. ChilePropiedades: Páginas 1 a 2

--- 📄 Página 1 ---
Página cargada: Departamentos en Arriendo en Coquimbo
Links encontrados: 95

--- 📄 Página 2 ---
Página cargada: Departamentos en Arriendo en Coquimbo
Links encontrados: 95

Catálogo listo: 10 propiedades
Procesando 1/10
Procesando 2/10
Procesando 3/10
Procesando 4/10
Procesando 5/10
Procesando 6/10
Procesando 7/10
Procesando 8/10
Procesando 9/10
Procesando 10/10
Guardando Mongo
Extracción terminada: 10


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ChilePropiedades_Analisis") \
    .config("spark.jars.packages","org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .config("spark.mongodb.input.uri","mongodb://database:27017/ArriendosPrueba_IVRegión.G2_PruebaChilePropiedades") \
    .config("spark.mongodb.output.uri","mongodb://database:27017/ArriendosPrueba_IVRegión.G2_PruebaChilePropiedades") \
    .getOrCreate()

df = spark.read.format("mongo").load()

df.show(5)

+--------------------+-----+-----------+---------------+-------------------+--------+----------+---+-------+--------+-------+----------------+-------+--------------------+---------+--------------------+
|                 _id|baños|dormitorios|estacionamiento|   fecha_extraccion|gimnasio|lavanderia| m2|piscina|  precio|quincho|     responsable|terraza|              titulo|ubicacion|                 url|
+--------------------+-----+-----------+---------------+-------------------+--------+----------+---+-------+--------+-------+----------------+-------+--------------------+---------+--------------------+
|{69e703b8515bfd34...|    2|          2|              1|2026-04-21 04:56:16|       0|         0| 75|      1|720000.0|      0|Constanza Torres|      0|Departamentos en ...| Coquimbo|https://chileprop...|
|{69e703b8515bfd34...|    0|          0|              1|2026-04-21 04:56:23|       0|         0| 50|      0|530000.0|      0|Constanza Torres|      0|Departamentos en ...| Coquimbo|https:/

In [2]:
#PARA VER ESTRUCTURA DE DATOS
df.printSchema()

root
 |-- _id: struct (nullable = true)
 |    |-- oid: string (nullable = true)
 |-- baños: integer (nullable = true)
 |-- dormitorios: integer (nullable = true)
 |-- estacionamiento: integer (nullable = true)
 |-- fecha_extraccion: string (nullable = true)
 |-- gimnasio: integer (nullable = true)
 |-- lavanderia: integer (nullable = true)
 |-- m2: integer (nullable = true)
 |-- piscina: integer (nullable = true)
 |-- precio: double (nullable = true)
 |-- quincho: integer (nullable = true)
 |-- responsable: string (nullable = true)
 |-- terraza: integer (nullable = true)
 |-- titulo: string (nullable = true)
 |-- ubicacion: string (nullable = true)
 |-- url: string (nullable = true)



In [3]:
#LIMPIAR DATOS BÁSICOS
from pyspark.sql.functions import col

df_limpio = df.filter(
    (col("precio") > 0) &
    (col("m2") > 0)
)

df_limpio.show(5)

+--------------------+-----+-----------+---------------+-------------------+--------+----------+---+-------+--------+-------+----------------+-------+--------------------+---------+--------------------+
|                 _id|baños|dormitorios|estacionamiento|   fecha_extraccion|gimnasio|lavanderia| m2|piscina|  precio|quincho|     responsable|terraza|              titulo|ubicacion|                 url|
+--------------------+-----+-----------+---------------+-------------------+--------+----------+---+-------+--------+-------+----------------+-------+--------------------+---------+--------------------+
|{69e703b8515bfd34...|    2|          2|              1|2026-04-21 04:56:16|       0|         0| 75|      1|720000.0|      0|Constanza Torres|      0|Departamentos en ...| Coquimbo|https://chileprop...|
|{69e703b8515bfd34...|    0|          0|              1|2026-04-21 04:56:23|       0|         0| 50|      0|530000.0|      0|Constanza Torres|      0|Departamentos en ...| Coquimbo|https:/

In [6]:
#Comparación por amenidades

df_analisis.groupBy("piscina").avg("precio").show()
df_analisis.groupBy("gimnasio").avg("precio").show()
df_analisis.groupBy("estacionamiento").avg("precio").show()

+-------+-----------+
|piscina|avg(precio)|
+-------+-----------+
|      1|   720000.0|
|      0|   570000.0|
+-------+-----------+

+--------+-----------+
|gimnasio|avg(precio)|
+--------+-----------+
|       1|   650000.0|
|       0|   587500.0|
+--------+-----------+

+---------------+-----------+
|estacionamiento|avg(precio)|
+---------------+-----------+
|              1|   600000.0|
+---------------+-----------+



In [8]:
#IMPACTO POR AMENIDADES

from pyspark.sql.functions import count, avg, round

print("--- IMPACTO DE AMENIDADES POR UBICACIÓN ---")

amenidades = [
    "piscina",
    "quincho",
    "gimnasio",
    "estacionamiento",
    "terraza",
    "lavanderia"
]

for am in amenidades:
    
    print(f"\n Análisis de: {am.upper()}")

    df.groupBy("ubicacion", am).agg(
        count("*").alias("Cantidad"),
        round(avg("precio"),0).alias("Precio_Promedio"),
        round(avg("m2"),0).alias("M2_Promedio")
    ).orderBy("ubicacion").show()

--- IMPACTO DE AMENIDADES POR UBICACIÓN ---

 Análisis de: PISCINA
+---------+-------+--------+---------------+-----------+
|ubicacion|piscina|Cantidad|Precio_Promedio|M2_Promedio|
+---------+-------+--------+---------------+-----------+
| Coquimbo|      0|       8|       570000.0|       57.0|
| Coquimbo|      1|       2|       720000.0|       75.0|
+---------+-------+--------+---------------+-----------+


 Análisis de: QUINCHO
+---------+-------+--------+---------------+-----------+
|ubicacion|quincho|Cantidad|Precio_Promedio|M2_Promedio|
+---------+-------+--------+---------------+-----------+
| Coquimbo|      0|      10|       600000.0|       60.0|
+---------+-------+--------+---------------+-----------+


 Análisis de: GIMNASIO
+---------+--------+--------+---------------+-----------+
|ubicacion|gimnasio|Cantidad|Precio_Promedio|M2_Promedio|
+---------+--------+--------+---------------+-----------+
| Coquimbo|       0|       8|       587500.0|       61.0|
| Coquimbo|       1|     

In [9]:
#COMPARACION POR DORMITORIOS

print("--- COMPARACIÓN POR DORMITORIOS ---")

df.groupBy("ubicacion", "dormitorios").agg(
    count("*").alias("Cantidad"),
    round(avg("precio"),0).alias("Precio_Promedio"),
    round(avg("m2"),0).alias("M2_Promedio")
).orderBy("ubicacion","dormitorios").show()

--- COMPARACIÓN POR DORMITORIOS ---
+---------+-----------+--------+---------------+-----------+
|ubicacion|dormitorios|Cantidad|Precio_Promedio|M2_Promedio|
+---------+-----------+--------+---------------+-----------+
| Coquimbo|          0|       2|       530000.0|       50.0|
| Coquimbo|          1|       2|       450000.0|       57.0|
| Coquimbo|          2|       6|       673333.0|       65.0|
+---------+-----------+--------+---------------+-----------+



In [11]:
#COMPARACION POR BAÑOR
print("--- COMPARACIÓN POR BAÑOS ---")

df.groupBy("ubicacion", "baños").agg(
    count("*").alias("Cantidad"),
    round(avg("precio"),0).alias("Precio_Promedio")
).orderBy("ubicacion","baños").show()

--- COMPARACIÓN POR BAÑOS ---
+---------+-----+--------+---------------+
|ubicacion|baños|Cantidad|Precio_Promedio|
+---------+-----+--------+---------------+
| Coquimbo|    0|       2|       530000.0|
| Coquimbo|    1|       4|       550000.0|
| Coquimbo|    2|       4|       685000.0|
+---------+-----+--------+---------------+



In [10]:
#COMPARACION POR ESTACIONAMIENTO
print("--- COMPARACIÓN ESTACIONAMIENTO ---")

df.groupBy("ubicacion", "estacionamiento").agg(
    count("*").alias("Cantidad"),
    round(avg("precio"),0).alias("Precio_Promedio")
).orderBy("ubicacion").show()

--- COMPARACIÓN ESTACIONAMIENTO ---
+---------+---------------+--------+---------------+
|ubicacion|estacionamiento|Cantidad|Precio_Promedio|
+---------+---------------+--------+---------------+
| Coquimbo|              1|      10|       600000.0|
+---------+---------------+--------+---------------+



In [4]:
#Crear precio por metro cuadrado

from pyspark.sql.functions import round

df_analisis = df_limpio.withColumn(
    "precio_m2",
    round(col("precio") / col("m2"), 2)
)

df_analisis.select(
    "titulo",
    "precio",
    "m2",
    "precio_m2"
).show()

+--------------------+--------+---+---------+
|              titulo|  precio| m2|precio_m2|
+--------------------+--------+---+---------+
|Departamentos en ...|720000.0| 75|   9600.0|
|Departamentos en ...|530000.0| 50|  10600.0|
|Departamentos en ...|450000.0| 57|  7894.74|
|Departamentos en ...|650000.0| 60| 10833.33|
|Departamentos en ...|650000.0| 60| 10833.33|
|Departamentos en ...|720000.0| 75|   9600.0|
|Departamentos en ...|530000.0| 50|  10600.0|
|Departamentos en ...|450000.0| 57|  7894.74|
|Departamentos en ...|650000.0| 60| 10833.33|
|Departamentos en ...|650000.0| 60| 10833.33|
+--------------------+--------+---+---------+



In [5]:
# Comparación por dormitorios

df_analisis.groupBy("dormitorios").avg(
    "precio",
    "m2",
    "precio_m2"
).orderBy("dormitorios").show()

+-----------+-----------------+-------+------------------+
|dormitorios|      avg(precio)|avg(m2)|    avg(precio_m2)|
+-----------+-----------------+-------+------------------+
|          0|         530000.0|   50.0|           10600.0|
|          1|         450000.0|   57.0|           7894.74|
|          2|673333.3333333334|   65.0|10422.220000000001|
+-----------+-----------------+-------+------------------+

